# 🔒 Local Test 003: Strict Zero Target Leakage Validator

This notebook audits our strict forward-only temporal chronological features.
It scientifically validates that future target labels NEVER leak backward to affect current predictions.

In [ ]:
import os
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

project_root = Path(os.getcwd()).resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))


print("✅ Leakage validator dependencies loaded successfully!")

In [ ]:
print("🔄 Running Chronological Forward-Only Target flow validator...")
# We simulate a future target label perturbation and check if past features change.
test_df = pd.DataFrame({
    'subject_id': ['user_1', 'user_1', 'user_1', 'user_1'],
    'date': ['2026-05-01', '2026-05-02', '2026-05-03', '2026-05-04'],
    'y_true': [1.0, 0.0, 1.0, 0.0]
})
print("Simulated clean sequence:")
print(test_df)

In [ ]:
def add_temporal_target_features_leakfree(df, y, train_idx, target_col, id_col, half_life=3.0):
    df_copy = df.copy()
    y_train = pd.Series(np.nan, index=df_copy.index)
    y_train.loc[train_idx] = y.loc[train_idx]
    df_copy['y_train_temp'] = y_train

    df_copy['date_temp'] = pd.to_datetime(df_copy['date'])
    df_copy = df_copy.sort_values(by=[id_col, 'date_temp']).reset_index(drop=True)

    df_copy['date_with_target'] = df_copy['date_temp'].where(df_copy['y_train_temp'].notna())
    df_copy['y_train_prev_source'] = df_copy.groupby(id_col)['y_train_temp'].shift(1)
    df_copy['date_prev_source'] = df_copy.groupby(id_col)['date_with_target'].shift(1)

    df_copy['y_prev'] = df_copy.groupby(id_col)['y_train_prev_source'].ffill()
    df_copy['date_prev'] = df_copy.groupby(id_col)['date_prev_source'].ffill()
    df_copy['delta_prev'] = (df_copy['date_temp'] - pd.to_datetime(df_copy['date_prev'])).dt.days

    subject_means = df_copy.groupby(id_col)['y_train_temp'].transform('mean').fillna(0.5)

    decay_lambda = np.log(2.0) / half_life
    w_prev = np.exp(-decay_lambda * df_copy['delta_prev'].fillna(999.0))

    prior_weight = 0.2
    denom = w_prev + prior_weight
    df_copy['y_interp'] = (w_prev * df_copy['y_prev'].fillna(0.5) + prior_weight * subject_means) / denom

    # STRICT FORWARD FORCE
    df_copy['y_prev'] = df_copy['y_prev'].fillna(subject_means)
    df_copy['y_next'] = subject_means
    df_copy['y_interp'] = df_copy['y_interp'].fillna(subject_means)

    features = pd.DataFrame(index=df.index)
    features['y_prev'] = df_copy['y_prev'].values
    features['y_next'] = df_copy['y_next'].values
    features['y_interp'] = df_copy['y_interp'].values
    return features

# Base Features
feats_orig = add_temporal_target_features_leakfree(test_df, test_df['y_true'], [0, 1, 2], 'y_true', 'subject_id')

# Perturb future index [3] (Change 0.0 -> 9.0)
test_df_perturbed = test_df.copy()
test_df_perturbed.loc[3, 'y_true'] = 9.0
feats_perturbed = add_temporal_target_features_leakfree(test_df_perturbed, test_df_perturbed['y_true'], [0, 1, 2, 3], 'y_true', 'subject_id')

# Check if index [0] to [2] features changed
diff = np.abs(feats_orig.iloc[:3]['y_interp'].values - feats_perturbed.iloc[:3]['y_interp'].values).sum()

print(f"\nTotal backward leakage difference: {diff:.6f}")
if diff == 0.0:
    print("🎉 100% SUCCESS! Zero Target Leakage verified. Past features remain mathematically immune to future target variations.")
else:
    print("🚨 LEAK DETECTED! Future perturbation corrupted past parameters.")